# Explore: Reviews

`olist_order_reviews_dataset.csv`

In [1]:
import pandas as pd

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv") # used later
reviews.shape

(99224, 7)

99224 rows, 7 columns.

In [2]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


Columns cover review and order ID, review comment title and message, review creation date and answer timestamp, and review score.

In [3]:
reviews.dtypes

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

All columns except `review_score` (which is `int64`) are `str`, even the date and time columns

In [4]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Missing values are just the review comment title and message fields. Title is blank 88% of the time, message 59% (both optional fields), score is always filled. Score without text is common.

In [5]:
counts = reviews["review_score"].value_counts()
shares = reviews["review_score"].value_counts(normalize = True).mul(100).round(1)
review_table = pd.DataFrame({"count": counts, "percent": shares}).sort_index()
review_table

,count,percent
review_score,,
1,11424,11.5
2,3151,3.2
3,8179,8.2
4,19142,19.3
5,57328,57.8


About 58% of reviews are scored 5 out of 5, ~20% are scored 4 and ~12% are scored 1.

In [6]:
counts = reviews["review_id"].value_counts()
review_id_frequency = counts.value_counts().sort_index()
review_id_frequency

count
1    97621
2      764
3       25
Name: count, dtype: int64

Most review IDs appear once. 764 appear twice and 25 three times. These are not copied rows: the same review (same score and text) is linked to two different orders.

In [7]:
review_counts = reviews["order_id"].value_counts()
multi_review_orders = (review_counts > 1).sum()
multi_review_orders

np.int64(547)

In [8]:
len(set(orders["order_id"]) - set(reviews["order_id"]))

768

547 orders have more than one review row. 768 orders were never reviewed.

In [9]:
multi_orders = reviews["order_id"].value_counts()
multi_orders = multi_orders[multi_orders > 1].index
sample_order = multi_orders[0]
reviews[reviews["order_id"] == sample_order]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21 00:00:00,2017-07-26 13:45:15


All rows for this one order have different review IDs. Scores can differ between rows, so an order can be reviewed more than once. For analysis, pick one row per order (for example the latest) or aggregate on purpose.